# ViHateT5 Demo - Inference on 3 Tasks

This notebook demonstrates how to use the ViHateT5 model for inference on 3 tasks:
- **ViHSD**: Hate Speech Detection (CLEAN, OFFENSIVE, HATE)
- **ViCTSD**: Toxic Speech Detection (NONE, TOXIC)
- **ViHOS**: Hate Spans Detection (highlight hate spans)


In [ ]:
# Import libraries
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer
import unicodedata

print(" Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load ViHateT5 Model
model_name = "Minhbao5xx2/balance_Vi_T5"  # Best model according to README

print(f"Loading model: {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print(f" Model loaded successfully on {device}!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")


## Task 1: ViHSD - Hate Speech Detection

Classify text as: **CLEAN**, **OFFENSIVE**, or **HATE**


In [ ]:
# ViHSD Inference
# Sample texts with corresponding labels (Vietnamese hate speech examples)
vihsd_samples = [
    ("Cảm ơn bạn đã chia sẻ thông tin hữu ích này!", "CLEAN"),
    ("Bạn nói chuyện như thế này thật không hay chút nào.", "OFFENSIVE"),
    ("Đồ ngu như mày thì đừng có nói nữa!", "HATE")
]

print("=" * 80)
print("ViHSD - Hate Speech Detection")
print("=" * 80)

for text, expected_label in vihsd_samples:
    # Format input with task prefix
    input_text = f"hate-speech-detection: {text}"
    
    # Tokenize
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=64,
            num_beams=1,
            do_sample=False
        )
    
    # Decode
    predicted_label = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    # Display results
    print(f"\n Text: {text}")
    print(f"   Expected: {expected_label}")
    print(f"   Predicted: {predicted_label}")
    print(f"    Match: {predicted_label.upper() == expected_label.upper()}")

print("\n" + "=" * 80)


## Task 2: ViCTSD - Toxic Speech Detection

Classify text as: **NONE** or **TOXIC**


In [ ]:
# ViCTSD Inference
# Sample texts with corresponding labels (Vietnamese toxic speech examples)
victsd_samples = [
    ("Bài viết này rất hay và có ý nghĩa.", "NONE"),
    ("Mày là đồ rác rưởi, đừng có xuất hiện ở đây nữa!", "TOXIC")
]

print("=" * 80)
print("ViCTSD - Toxic Speech Detection")
print("=" * 80)

for text, expected_label in victsd_samples:
    # Format input with task prefix
    input_text = f"toxic-speech-detection: {text}"
    
    # Tokenize
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=64,
            num_beams=1,
            do_sample=False
        )
    
    # Decode
    predicted_label = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    # Display results
    print(f"\n Text: {text}")
    print(f"   Expected: {expected_label}")
    print(f"   Predicted: {predicted_label}")
    print(f"    Match: {predicted_label.upper() == expected_label.upper()}")

print("\n" + "=" * 80)


## Task 3: ViHOS - Hate Spans Detection

Detect and highlight hate speech spans using `[HATE]...[HATE]` tags (same tag for both start and end)


In [ ]:
# ViHOS Inference
# Sample text with hate spans (Vietnamese)
vihos_sample = "Mày là đồ ngu ngốc, đừng có nói nữa!"

print("=" * 80)
print("ViHOS - Hate Spans Detection")
print("=" * 80)

# Format input with task prefix
input_text = f"hate-spans-detection: {vihos_sample}"

# Tokenize
inputs = tokenizer(
    input_text,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=256
).to(device)

# Generate
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=256,
        num_beams=1,
        do_sample=False
    )

# Decode
predicted_output = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

# Display results
print(f"\nOriginal Text: {vihos_sample}")
print(f"\nPredicted Output (with [HATE] tags):")
print(f"   {predicted_output}")

# Extract hate spans if present - Format: [HATE]text[HATE] (same tag for both start and end)
if "[HATE]" in predicted_output or "[hate]" in predicted_output.lower():
    print(f"\n Model detected hate spans!")
    
    # Extract span indices - following the same logic as train_t5.py
    def find_and_extract_substrings(original_str, input_str):
        """Extract character indices from generated text with [HATE] tags.
        Format: [HATE]text[HATE] (same tag for both start and end)
        """
        start_tag = '[hate]'
        end_tag = '[hate]'  # Same tag
        
        input_str_norm = unicodedata.normalize('NFC', input_str.lower())
        original_str_norm = unicodedata.normalize('NFC', original_str.lower())
        
        substrings = []
        start_index = input_str_norm.find(start_tag)
        while start_index != -1:
            end_index = input_str_norm.find(end_tag, start_index + len(start_tag))
            if end_index != -1:
                substrings.append(input_str_norm[start_index + len(start_tag):end_index])
                start_index = input_str_norm.find(start_tag, end_index + len(end_tag))
            else:
                break
        
        if not substrings:
            return []
        
        indices_list = []
        for substring in substrings:
            start_idx = original_str_norm.find(substring)
            while start_idx != -1:
                indices_list.extend(list(range(start_idx, start_idx + len(substring))))
                start_idx = original_str_norm.find(substring, start_idx + 1)
        
        return sorted(set(indices_list))
    
    spans_indices = find_and_extract_substrings(vihos_sample, predicted_output)
    if spans_indices:
        print(f"\n Hate span character indices: {spans_indices}")
        if spans_indices:
            span_text = vihos_sample[min(spans_indices):max(spans_indices)+1]
            print(f"   Highlighted text: '{span_text}'")
else:
    print(f"\n  Model did not detect any hate spans in this text.")

print("\n" + "=" * 80)
